# TRELLIS.2 — Drive Batch Image → 3D

**Plug-and-play:** choose a GPU runtime and use **Runtime → Run all**.

Upload images to:
`MyDrive/TRELLIS_INPUT`

Generated `.glb` files are saved to:
`MyDrive/TRELLIS_OUTPUT`

Successfully processed images move to `MyDrive/TRELLIS_DONE`; failed inputs move to `MyDrive/TRELLIS_FAILED`.

The TRELLIS model loads once and processes the entire input folder automatically. PNG, JPG, JPEG and WEBP are supported.

> One-time requirement: Colab Secrets should contain `HF_TOKEN` for the gated TRELLIS dependencies. If it is missing, the notebook asks for it securely.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, subprocess, getpass

# 1) Mount Drive and create the only folders you need.
drive.mount("/content/drive", force_remount=False)

INPUT_DIR  = Path("/content/drive/MyDrive/TRELLIS_INPUT")
OUTPUT_DIR = Path("/content/drive/MyDrive/TRELLIS_OUTPUT")
DONE_DIR   = Path("/content/drive/MyDrive/TRELLIS_DONE")
FAILED_DIR = Path("/content/drive/MyDrive/TRELLIS_FAILED")
CACHE_DIR  = Path("/content/drive/MyDrive/AI3D_Engine_Cache")

for folder in (INPUT_DIR, OUTPUT_DIR, DONE_DIR, FAILED_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Upload images here :", INPUT_DIR)
print("3D models save here:", OUTPUT_DIR)

# 2) Get the maintained TRELLIS helpers.
REPO = Path("/content/My-works")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/Logan17de/My-works.git", str(REPO)],
    check=True,
)

TOOLS_3D = REPO / "ai-3d-animation-engines" / "3d-engine"

# Reuse compiled/download cache between Colab sessions.
os.environ["ENGINE_CACHE_ROOT"] = str(CACHE_DIR)

# 3) Install TRELLIS.2.
subprocess.run(["bash", str(TOOLS_3D / "install_3d.sh")], check=True)

# 4) Hugging Face auth + model preparation.
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: " ).strip()
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets and run again.")

env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
env["HF_HOME"] = "/content/huggingface"
env["HF_XET_HIGH_PERFORMANCE"] = "1"
env["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [
        "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
        "python", str(TOOLS_3D / "prepare_hf_models.py"),
    ],
    cwd="/content/TRELLIS.2",
    env=env,
    check=True,
)

print("\n✅ Setup complete.")
print(f"Put PNG/JPG/JPEG/WEBP images in: {INPUT_DIR}")


In [ ]:
# Process every image in TRELLIS_INPUT.
# TRELLIS loads once, each successful image becomes a .glb, and the input is moved to TRELLIS_DONE.

from pathlib import Path
import os, subprocess

BATCH_RUNNER = Path("/content/trellis_drive_batch.py")
BATCH_RUNNER.write_text('from __future__ import annotations\n\nimport os\nimport shutil\nimport sys\nimport traceback\nfrom pathlib import Path\n\nos.environ.setdefault("OPENCV_IO_ENABLE_OPENEXR", "1")\nos.environ.setdefault("HF_HOME", "/content/huggingface")\n\nTRELLIS_ROOT = Path("/content/TRELLIS.2")\nsys.path.insert(0, str(TRELLIS_ROOT))\n\nimport torch\nfrom PIL import Image\nimport o_voxel\nfrom trellis2.pipelines import Trellis2ImageTo3DPipeline\n\nINPUT_DIR  = Path("/content/drive/MyDrive/TRELLIS_INPUT")\nOUTPUT_DIR = Path("/content/drive/MyDrive/TRELLIS_OUTPUT")\nDONE_DIR   = Path("/content/drive/MyDrive/TRELLIS_DONE")\nFAILED_DIR = Path("/content/drive/MyDrive/TRELLIS_FAILED")\n\nEXTS = {".png", ".jpg", ".jpeg", ".webp"}\n\ndef move_replace(src: Path, folder: Path) -> None:\n    dst = folder / src.name\n    if dst.exists():\n        dst.unlink()\n    shutil.move(str(src), str(dst))\n\nimages = sorted(p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXTS)\n\nif not images:\n    print("✅ No images waiting in TRELLIS_INPUT.")\n    raise SystemExit(0)\n\nprint(f"Found {len(images)} image(s). Loading TRELLIS.2 once...")\n\npipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")\npipeline.cuda()\n\nprint("✅ Model ready. Starting batch.\\n")\n\nsuccess = 0\nfailed = 0\n\nfor i, image_path in enumerate(images, 1):\n    output_path = OUTPUT_DIR / f"{image_path.stem}.glb"\n    print(f"[{i}/{len(images)}] {image_path.name}")\n\n    try:\n        if output_path.exists() and output_path.stat().st_size > 0:\n            print("  ↳ GLB already exists; marking input done.")\n            move_replace(image_path, DONE_DIR)\n            success += 1\n            continue\n\n        with Image.open(image_path) as im:\n            image = im.copy()\n\n        mesh = pipeline.run(image)[0]\n        mesh.simplify(16_777_216)\n\n        glb = o_voxel.postprocess.to_glb(\n            vertices=mesh.vertices,\n            faces=mesh.faces,\n            attr_volume=mesh.attrs,\n            coords=mesh.coords,\n            attr_layout=mesh.layout,\n            voxel_size=mesh.voxel_size,\n            aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n            decimation_target=500_000,\n            texture_size=2048,\n            remesh=True,\n            remesh_band=1,\n            remesh_project=0,\n            verbose=False,\n        )\n\n        glb.export(str(output_path), extension_webp=False)\n\n        if not output_path.is_file() or output_path.stat().st_size == 0:\n            raise RuntimeError("GLB export failed.")\n\n        move_replace(image_path, DONE_DIR)\n        success += 1\n        print(f"  ✅ Saved: {output_path}")\n\n        del image, mesh, glb\n        torch.cuda.empty_cache()\n\n    except Exception as exc:\n        failed += 1\n        print(f"  ❌ Failed: {exc}")\n        traceback.print_exc()\n        try:\n            move_replace(image_path, FAILED_DIR)\n        except Exception:\n            pass\n        torch.cuda.empty_cache()\n\nprint("\\n==============================")\nprint(f"✅ Done:   {success}")\nprint(f"❌ Failed: {failed}")\nprint(f"📁 GLBs:   {OUTPUT_DIR}")\nprint("==============================")', encoding="utf-8")

env = os.environ.copy()
env["HF_HOME"] = "/content/huggingface"
env["HF_XET_HIGH_PERFORMANCE"] = "1"
if "HF_TOKEN" not in env:
    env["HF_TOKEN"] = HF_TOKEN

subprocess.run(
    [
        "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
        "python", str(BATCH_RUNNER),
    ],
    cwd="/content/TRELLIS.2",
    env=env,
    check=True,
)
